In [1]:
import torch
import torch.nn as nn
import torchmetrics
import torchvision.transforms.v2 as T
import torchvision

from torch.utils.data import DataLoader
import torch.nn.functional as F

In [2]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(dtype=torch.float32, scale=True)])

train_valid_set = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True, transform=toTensor)
test_set = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True, transform=toTensor)

torch.manual_seed(52)
train_set, valid_set = torch.utils.data.random_split(train_valid_set, [45000, 5000])

/home/damian/test/test/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
train_data = DataLoader(train_set, shuffle=True, batch_size=64, pin_memory=True, num_workers=8, persistent_workers=True)
valid_data = DataLoader(valid_set, shuffle=True, batch_size=64)
test_data  = DataLoader(test_set, shuffle=True, batch_size=64)

In [4]:
model = nn.Sequential(nn.Flatten(), nn.Linear(32*32*3, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 10)).to('cuda')

In [5]:
def he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight.data)
        torch.zero_(module.bias.data)
        
model.apply(he_init)
print('done')

done


In [6]:
optimizer = torch.optim.NAdam(params=model.parameters(), lr=0.0005, momentum_decay=0.02, betas=(0.9, 0.99))
xentropy = nn.CrossEntropyLoss()

In [7]:
def train(model, optimizer, criterion, data_loader, n_epochs, early_stop=3):
    model.train()
    
    loss_list = []
    eps = 0.0005
    es_count = 0
    
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
        
        mean_loss = total_loss / len(data_loader)
        print(f'epoch: {epoch}, loss: {mean_loss}')
        
        loss_list.append(mean_loss)
        if epoch > 2:
            if abs(loss_list[-1] - loss_list[-2]) < eps:
                es_count +=1
                if es_count == early_stop:
                    print('Early stop!')
                    break

In [8]:
def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        
    return metric.compute()

In [ ]:
#c.
train(model, optimizer, xentropy, train_data, 10)

In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate(model, valid_data, accuracy)

tensor(0.4732, device='cuda:0')

## d

In [ ]:
model_d = nn.Sequential(nn.Flatten(), nn.BatchNorm1d(32*32*3), nn.Linear(32*32*3, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 100), nn.SiLU(),
                      nn.Linear(100, 10)).to('cuda')

In [ ]:
#d. twice faster and approximately the same result OUTDATED
#twice faster but started doing much worse OUTDATED
# if forgot he_init, after applying it converged to the same result twice faster (time and epochs)
model_d.apply(he_init)
print('done')


optimizer_d = torch.optim.NAdam(params=model_d.parameters(), lr=0.0005, momentum_decay=0.02, betas=(0.9, 0.99))
xentropy = nn.CrossEntropyLoss()

train(model_d, optimizer_d, xentropy, train_data, n_epochs=10)

done
epoch: 0, loss: 1.877225557511503
epoch: 1, loss: 1.643982482227412
epoch: 2, loss: 1.5502110033888707
epoch: 3, loss: 1.4787596205080098
epoch: 4, loss: 1.421316499737176
epoch: 5, loss: 1.3735475692559371
epoch: 6, loss: 1.326364852742038
epoch: 7, loss: 1.2885445995594969
epoch: 8, loss: 1.2583007504316894
epoch: 9, loss: 1.2219635058533063


In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate(model_d, valid_data, accuracy)

tensor(0.4928, device='cuda:0')

## e

In [9]:
model_e = nn.Sequential(nn.Flatten(), nn.Linear(32*32*3, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 100), nn.SELU(),
                      nn.Linear(100, 10)).to('cuda')

In [10]:
import math

def lecun_init(module):
    if isinstance(module, nn.Linear):
        fan_in = module.weight.size(1)
        std = 1 / math.sqrt(fan_in)
        nn.init.normal_(module.weight.data, mean=0, std=std)
        torch.zero_(module.bias.data)
        
model_e.apply(lecun_init)
print('done')

done


In [11]:
toTensor_e = T.Compose([T.ToImage(), T.ToDtype(dtype=torch.float32, scale=True), T.Normalize(mean=[0,0,0], std=[1,1,1])])

train_valid_set_e = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True, transform=toTensor_e)
test_set_e = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True, transform=toTensor_e)

torch.manual_seed(52)
train_set_e, valid_set_e = torch.utils.data.random_split(train_valid_set_e, [45000, 5000])

train_data_e = DataLoader(train_set_e, shuffle=True, batch_size=64, pin_memory=True, num_workers=8, persistent_workers=True)
valid_data_e = DataLoader(valid_set_e, shuffle=True, batch_size=64)
test_data_e  = DataLoader(test_set_e, shuffle=True, batch_size=64)

In [ ]:
# did not outperform previous ones
optimizer_e = torch.optim.NAdam(params=model_e.parameters(), lr=0.0005, momentum_decay=0.02, betas=(0.9, 0.99))
xentropy = nn.CrossEntropyLoss()

train(model_e, optimizer_e, xentropy, train_data_e, n_epochs=10)

epoch: 0, loss: 2.0268117258833214
epoch: 1, loss: 1.7922314644198527
epoch: 2, loss: 1.689248293468898
epoch: 3, loss: 1.621689667586576
epoch: 4, loss: 1.5707778024741195
epoch: 5, loss: 1.5335947715423324
epoch: 6, loss: 1.5003322796388106
epoch: 7, loss: 1.4698952207849785
epoch: 8, loss: 1.4447029883211309
epoch: 9, loss: 1.4145691858773881


In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate(model_e, valid_data_e, accuracy)

tensor(0.4462, device='cuda:0')

## f

In [32]:
model_f = nn.Sequential(nn.Flatten(), nn.AlphaDropout(0.1), nn.Linear(32*32*3, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 100), nn.SELU(),
                      nn.AlphaDropout(0.1), nn.Linear(100, 10)).to('cuda')

model_f.apply(lecun_init)
print('done')

done


In [ ]:
optimizer_f = torch.optim.NAdam(params=model_f.parameters(), lr=0.0005, momentum_decay=0.02, betas=(0.9, 0.99))
xentropy = nn.CrossEntropyLoss()

train(model_f, optimizer_f, xentropy, train_data_e, n_epochs=10)

epoch: 0, loss: 2.3652188019319014
epoch: 1, loss: 2.1321395484899934
epoch: 2, loss: 2.0993664916604757
epoch: 3, loss: 2.0780467204749584
epoch: 4, loss: 2.0602217873727735
epoch: 5, loss: 2.039657982235605
epoch: 6, loss: 2.0279736583205787
epoch: 7, loss: 1.9992165760221807
epoch: 8, loss: 1.9859776510433718
epoch: 9, loss: 1.9588160032237119


In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate(model_f, valid_data_e, accuracy)

tensor(0.2410, device='cuda:0')

In [ ]:
## this module can be implemented instead of dropout, but task is different
class MCDropout(nn.Dropout):
    def forward(self, input):
        return F.dropout(input, self.p, training=True)

In [13]:
def evaluate_mc(model, data_loader, metric):
    model.eval()
    metric.reset()
    
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            batch_size = X_batch.size(0)
            X_batch_new = X_batch.repeat_interleave(10, dim=0).to('cuda')
            y_batch = y_batch.to('cuda')
            #print(y_batch.size())
            y_pred = (model(X_batch_new).reshape(batch_size, 10, 10)).mean(dim=1)
            metric.update(y_pred, y_batch)
        
    return metric.compute()

In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate_mc(model_f, valid_data_e, accuracy)

tensor(0.2410, device='cuda:0')

## g

In [33]:
optimizer_g = torch.optim.NAdam(params=model_f.parameters(), lr=0.0005, momentum_decay=0.02, betas=(0.9, 0.99))
one_cycle = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer_g, max_lr=0.001, total_steps=10*len(train_data_e))

def train_scheduler(model, optimizer, scheduler, criterion, data_loader, n_epochs, early_stop=3):
    model.train()
    
    loss_list = []
    eps = 0.0005
    es_count = 0
    
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=1)
            
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
        
        mean_loss = total_loss / len(data_loader)
        print(f'epoch: {epoch}, loss: {mean_loss}')
        
        loss_list.append(mean_loss)
        if epoch > 2:
            if abs(loss_list[-1] - loss_list[-2]) < eps:
                es_count +=1
                if es_count == early_stop:
                    print('Early stop!')
                    break

In [34]:
xentropy = nn.CrossEntropyLoss()
train_scheduler(model_f, optimizer_g, one_cycle, xentropy, train_data_e, n_epochs=10)

epoch: 0, loss: 2.535241940820759
epoch: 1, loss: 2.2876842472363603
epoch: 2, loss: 2.1293002605776894
epoch: 3, loss: 2.104305739768527
epoch: 4, loss: 2.078100250356577
epoch: 5, loss: 2.0568620513447304
epoch: 6, loss: 2.027916320684281
epoch: 7, loss: 2.007105532017621
epoch: 8, loss: 1.9751372340727935
epoch: 9, loss: 1.9775898168710144


In [36]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
evaluate_mc(model_f, valid_data_e, accuracy)

tensor(0.2188, device='cuda:0')